# 03 — EDA Engine
**AutoAnalyst | Finance Module**

Performs domain-aware exploratory data analysis based on dataset type.
- `timeseries`    → returns, rolling averages, volatility, trend, volume analysis
- `transactional` → spend patterns, category breakdown, time trends, anomalies
- `fundamental`   → financial ratios, sector comparison, growth analysis

Input  : cleaned `df_clean` from `02_data_cleaning.ipynb`
Output : `eda_summary` dictionary — fed directly into `05_llm_insights.ipynb`

In [23]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded.')

✅ Libraries loaded.


In [24]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
DATASET_FILENAME = 'Annual_P_L_1_final.csv'
BASE_PATH        = r'R:\AutoAnalyst\finance_module\datasets'
DATASET_PATH     = os.path.join(BASE_PATH, DATASET_FILENAME)

print(f'Dataset : {DATASET_FILENAME}')
print(f'Exists  : {os.path.exists(DATASET_PATH)}')

Dataset : Annual_P_L_1_final.csv
Exists  : True


In [25]:
# ── Cell 3: Load + Clean (self-contained) ─────────────────────────────────────
# Each notebook is self-contained.
# We re-run the loader and cleaner here so this notebook works independently.

def smart_load(filepath):
    raw = pd.read_csv(filepath, nrows=3, header=0)
    first_val = str(raw.iloc[0, 0]).strip().lower()
    if first_val in ['ticker', 'date', 'symbol', 'name', 'description']:
        df = pd.read_csv(filepath, skiprows=[1, 2], header=0)
        df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
    else:
        df = pd.read_csv(filepath, header=0)
    return df

def detect_finance_type(df):
    cols_str = ' '.join([c.lower().strip() for c in df.columns])
    scores = {
        'timeseries'   : sum(1 for kw in ['close','open','high','low','volume','price'] if kw in cols_str),
        'transactional': sum(1 for kw in ['amount','card','exp type','city','gender'] if kw in cols_str),
        'fundamental'  : sum(1 for kw in ['sales','profit','eps','bse','nse','market cap','ratio','ebitda'] if kw in cols_str)
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'

def universal_clean(df):
    df = df.copy()
    df.drop_duplicates(inplace=True)
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].str.strip()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace(r'[^\w]', '_', regex=True))
    return df

def clean_timeseries(df):
    df = df.copy()
    date_col = 'date' if 'date' in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df.dropna(subset=[date_col], inplace=True)
    df.sort_values(date_col, inplace=True)
    df.reset_index(drop=True, inplace=True)
    num_cols = [c for c in df.columns if c != date_col]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df[num_cols] = df[num_cols].ffill()
    if 'close' in df.columns:
        Q1, Q3 = df['close'].quantile(0.25), df['close'].quantile(0.75)
        IQR = Q3 - Q1
        df['is_price_outlier'] = ((df['close'] < Q1 - 3*IQR) | (df['close'] > Q3 + 3*IQR))
    return df

def clean_transactional(df):
    df = df.copy()
    date_col = next((c for c in df.columns if 'date' in c.lower()), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
        df.sort_values(date_col, inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['month']       = df[date_col].dt.month
        df['year']        = df[date_col].dt.year
        df['day_of_week'] = df[date_col].dt.day_name()
    city_col = next((c for c in df.columns if 'city' in c.lower()), None)
    if city_col:
        df[city_col] = df[city_col].str.replace(', India', '', regex=False).str.strip()
    for col in ['exp_type', 'card_type', 'gender']:
        if col in df.columns:
            df[col] = df[col].str.title()
    if 'index' in df.columns:
        df.drop(columns=['index'], inplace=True)
    amount_col = next((c for c in df.columns if 'amount' in c.lower()), None)
    if amount_col:
        Q1, Q3 = df[amount_col].quantile(0.25), df[amount_col].quantile(0.75)
        IQR = Q3 - Q1
        df['is_amount_outlier'] = ((df[amount_col] < Q1 - 3*IQR) | (df[amount_col] > Q3 + 3*IQR))
    return df

def clean_fundamental(df):
    df = df.copy()
    if 'join_key' in df.columns:
        df.drop(columns=['join_key'], inplace=True)
    missing_pct  = df.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > 0.6].index.tolist()
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
    non_numeric = ['name', 'nse_code', 'industry', 'nse code', 'bse code']
    for col in [c for c in df.columns if c not in non_numeric and df[c].dtype == object]:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() > df[col].notna().sum() * 0.5:
            df[col] = converted
    key_cols = [c for c in ['sales','net_profit','profit_after_tax','market_capitalization','eps'] if c in df.columns]
    if key_cols:
        df['all_metrics_missing'] = df[key_cols].isnull().all(axis=1)
    return df

# ── Run pipeline ──────────────────────────────────────────────────────────────
df_raw       = smart_load(DATASET_PATH)
dataset_type = detect_finance_type(df_raw)
df_clean     = universal_clean(df_raw)

if dataset_type == 'timeseries':
    df_clean = clean_timeseries(df_clean)
elif dataset_type == 'transactional':
    df_clean = clean_transactional(df_clean)
elif dataset_type == 'fundamental':
    df_clean = clean_fundamental(df_clean)

print(f'✅ Loaded and cleaned: {DATASET_FILENAME}')
print(f'   Type  : {dataset_type.upper()}')
print(f'   Shape : {df_clean.shape}')

✅ Loaded and cleaned: Annual_P_L_1_final.csv
   Type  : FUNDAMENTAL
   Shape : (4668, 58)


In [26]:
# ── Cell 4: Descriptive Statistics ───────────────────────────────────────────
# Universal — runs on all dataset types.
# Gives a statistical overview of every numeric column.

print('=' * 60)
print('  DESCRIPTIVE STATISTICS')
print('=' * 60)

desc = df_clean.describe(include='all').T
display(desc)

# Store for EDA summary
desc_stats = df_clean.select_dtypes(include=[np.number]).describe().to_dict()

  DESCRIPTIVE STATISTICS


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
name,4668,4668,20 Microns,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bse_code,4668.0,NaN,NaN,NaN,475480.346187,158391.99505,-1.0,511311.25,530908.0,538996.25,590134.0
nse_code,4668,2355,__NA__,2314,NaN,NaN,NaN,NaN,NaN,NaN,NaN
industry,4668,107,Finance & Investments,560,NaN,NaN,NaN,NaN,NaN,NaN,NaN
current_price,4667.0,NaN,NaN,NaN,602.739565,2985.703112,0.03,27.0,103.24,385.625,128022.4
sales,4664.0,NaN,NaN,NaN,3676.532614,30030.073653,-20.2,13.3125,107.82,731.8025,901064.0
opm,4367.0,NaN,NaN,NaN,-89.535244,2264.93414,-102800.0,2.78,9.91,19.28,3094.12
profit_after_tax,4664.0,NaN,NaN,NaN,323.653407,2774.746549,-31973.31,0.08,4.13,40.45,69621.0
return_on_capital_employed,4451.0,NaN,NaN,NaN,15.896385,293.495726,-1535.0,2.49,9.6,18.655,18600.0
eps,4640.0,NaN,NaN,NaN,20.6845,193.525548,-486.41,0.0575,2.7,12.0525,8787.0


In [27]:
# ── Cell 5: Correlation Matrix ────────────────────────────────────────────────
# Universal — shows relationships between all numeric columns.
# High correlation = columns move together.
# For OHLCV data, Open/High/Low/Close should be highly correlated (~0.99).
# For fundamentals, interesting to find what correlates with profit.

num_df = df_clean.select_dtypes(include=[np.number])

# Drop flag columns from correlation (not meaningful)
flag_cols = [c for c in num_df.columns if c.startswith('is_') or c.endswith('_missing')]
num_df = num_df.drop(columns=flag_cols, errors='ignore')

corr = num_df.corr()

print('=' * 60)
print('  CORRELATION MATRIX')
print('=' * 60)
display(corr.round(3))

# Find top correlated pairs (excluding self-correlation)
print('\n  Top 10 strongest correlations:')
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['col_a', 'col_b', 'correlation']
corr_pairs['abs_corr'] = corr_pairs['correlation'].abs()
top_corr = corr_pairs.sort_values('abs_corr', ascending=False).head(10)
display(top_corr[['col_a', 'col_b', 'correlation']].reset_index(drop=True))

# Store for EDA summary
top_correlations = top_corr[['col_a', 'col_b', 'correlation']].values.tolist()

  CORRELATION MATRIX


,bse_code,current_price,sales,opm,profit_after_tax,return_on_capital_employed,eps,change_in_promoter_holding,sales_last_year,operating_profit_last_year,...,profit_after_tax_preceding_year,extraordinary_items_preceding_year,net_profit_preceding_year,dividend_preceding_year,opm_preceding_year,npm_preceding_year,eps_preceding_year,sales_preceding_12months,net_profit_preceding_12months,market_capitalization
bse_code,1.000,0.029,0.032,-0.016,0.031,-0.007,0.016,0.102,0.032,0.034,...,0.029,-0.000,0.029,0.021,-0.027,-0.007,-0.103,0.014,0.014,0.043
current_price,0.029,1.000,0.045,0.010,0.058,0.004,0.498,0.011,0.045,0.042,...,0.053,0.012,0.054,0.040,0.019,0.006,0.172,0.042,0.055,0.111
sales,0.032,0.045,1.000,0.006,0.819,0.002,0.029,0.008,1.000,0.753,...,0.718,-0.078,0.715,0.378,0.012,0.003,0.008,1.000,0.830,0.691
opm,-0.016,0.010,0.006,1.000,0.009,0.022,0.009,-0.002,0.006,0.008,...,0.007,0.001,0.007,0.005,0.301,-0.013,0.005,0.007,0.009,0.009
profit_after_tax,0.031,0.058,0.819,0.009,1.000,0.003,0.042,0.013,0.819,0.885,...,0.937,-0.056,0.923,0.542,0.014,0.004,0.015,0.812,0.986,0.830
return_on_capital_employed,-0.007,0.004,0.002,0.022,0.003,1.000,0.004,-0.000,0.002,0.000,...,0.003,0.000,0.003,0.004,0.181,0.079,0.008,0.013,0.017,0.003
eps,0.016,0.498,0.029,0.009,0.042,0.004,1.000,0.008,0.029,0.030,...,0.034,0.008,0.035,0.019,0.011,0.010,0.298,0.030,0.046,0.043
change_in_promoter_holding,0.102,0.011,0.008,-0.002,0.013,-0.000,0.008,1.000,0.008,0.006,...,0.014,0.001,0.014,0.004,-0.011,0.004,-0.039,0.003,0.009,0.010
sales_last_year,0.032,0.045,1.000,0.006,0.819,0.002,0.029,0.008,1.000,0.753,...,0.718,-0.078,0.715,0.378,0.012,0.003,0.008,1.000,0.830,0.691
operating_profit_last_year,0.034,0.042,0.753,0.008,0.885,0.000,0.030,0.006,0.753,1.000,...,0.834,-0.057,0.838,0.480,0.015,0.003,0.010,0.747,0.882,0.763



  Top 10 strongest correlations:


,col_a,col_b,correlation
0,interest_last_year,interest,1.000000
1,sales,sales_last_year,0.999996
2,operating_profit_last_year,operating_profit,0.999994
3,other_income_last_year,other_income,0.999993
4,ebit_last_year,ebit,0.999993
5,depreciation_last_year,depreciation,0.999992
6,tax_last_year,tax,0.999990
7,eps,eps_last_year,0.999955
8,net_profit_last_year,net_profit,0.999944
9,profit_after_tax,profit_after_tax_last_year,0.999937


In [28]:
# ── Cell 6: Type-Specific EDA ─────────────────────────────────────────────────
# Each dataset type gets its own domain-relevant analysis.

type_specific_summary = {}

# ════════════════════════════════════════════════════════════
# TIMESERIES EDA
# ════════════════════════════════════════════════════════════
if dataset_type == 'timeseries':
    print('=' * 60)
    print('  TIMESERIES EDA — RELIANCE STOCK')
    print('=' * 60)

    # ── Daily Returns ────────────────────────────────────────
    # Pct change from one day to next — core metric in finance
    df_clean['daily_return'] = df_clean['close'].pct_change() * 100

    avg_return  = df_clean['daily_return'].mean()
    std_return  = df_clean['daily_return'].std()
    best_day    = df_clean.loc[df_clean['daily_return'].idxmax()]
    worst_day   = df_clean.loc[df_clean['daily_return'].idxmin()]
    positive_days = (df_clean['daily_return'] > 0).sum()
    negative_days = (df_clean['daily_return'] < 0).sum()

    print(f'\n  Daily Returns:')
    print(f'    Average daily return : {avg_return:.4f}%')
    print(f'    Std dev (volatility)  : {std_return:.4f}%')
    print(f'    Best day  : {best_day["date"].date()} → +{best_day["daily_return"]:.2f}%')
    print(f'    Worst day : {worst_day["date"].date()} → {worst_day["daily_return"]:.2f}%')
    print(f'    Positive days : {positive_days} | Negative days : {negative_days}')

    # ── Rolling Averages ─────────────────────────────────────
    # Moving averages smooth out noise, reveal trend direction
    df_clean['ma_7']  = df_clean['close'].rolling(window=7).mean()
    df_clean['ma_30'] = df_clean['close'].rolling(window=30).mean()
    df_clean['ma_90'] = df_clean['close'].rolling(window=90).mean()

    latest = df_clean.iloc[-1]
    print(f'\n  Latest Price vs Moving Averages:')
    print(f'    Current price : ₹{latest["close"]:.2f}')
    print(f'    MA 7          : ₹{latest["ma_7"]:.2f}')
    print(f'    MA 30         : ₹{latest["ma_30"]:.2f}')
    print(f'    MA 90         : ₹{latest["ma_90"]:.2f}')

    # Trend signal: price vs MA 30
    trend = 'BULLISH 📈' if latest['close'] > latest['ma_30'] else 'BEARISH 📉'
    print(f'    Trend signal  : {trend}')

    # ── Volatility ───────────────────────────────────────────
    # Rolling 30-day annualized volatility
    df_clean['volatility_30d'] = df_clean['daily_return'].rolling(30).std() * np.sqrt(252)
    avg_vol = df_clean['volatility_30d'].mean()
    max_vol = df_clean['volatility_30d'].max()
    max_vol_date = df_clean.loc[df_clean['volatility_30d'].idxmax(), 'date']

    print(f'\n  Volatility (annualized):')
    print(f'    Average : {avg_vol:.2f}%')
    print(f'    Peak    : {max_vol:.2f}% on {max_vol_date.date()}')

    # ── Price Range Summary ──────────────────────────────────
    all_time_high = df_clean['high'].max()
    all_time_low  = df_clean['low'].min()
    ath_date      = df_clean.loc[df_clean['high'].idxmax(), 'date']
    atl_date      = df_clean.loc[df_clean['low'].idxmin(), 'date']
    total_return  = ((df_clean['close'].iloc[-1] - df_clean['close'].iloc[0]) /
                      df_clean['close'].iloc[0]) * 100

    print(f'\n  Price Range:')
    print(f'    All-time high : ₹{all_time_high:.2f} on {ath_date.date()}')
    print(f'    All-time low  : ₹{all_time_low:.2f} on {atl_date.date()}')
    print(f'    Total return (full period) : {total_return:.2f}%')

    # ── Volume Analysis ──────────────────────────────────────
    avg_vol_shares = df_clean['volume'].mean()
    max_vol_shares = df_clean['volume'].max()
    max_vol_date2  = df_clean.loc[df_clean['volume'].idxmax(), 'date']

    print(f'\n  Volume Analysis:')
    print(f'    Avg daily volume : {avg_vol_shares:,.0f} shares')
    print(f'    Peak volume      : {max_vol_shares:,.0f} shares on {max_vol_date2.date()}')

    # Volume-price correlation
    vol_price_corr = df_clean['volume'].corr(df_clean['daily_return'])
    print(f'    Volume-Return correlation : {vol_price_corr:.4f}')

    # ── Yearly Summary ───────────────────────────────────────
    df_clean['year'] = df_clean['date'].dt.year
    yearly = df_clean.groupby('year').agg(
        open_price  = ('open',  'first'),
        close_price = ('close', 'last'),
        high_price  = ('high',  'max'),
        low_price   = ('low',   'min'),
        avg_volume  = ('volume','mean')
    )
    yearly['annual_return_%'] = ((yearly['close_price'] - yearly['open_price']) /
                                   yearly['open_price'] * 100).round(2)
    print(f'\n  Yearly Summary:')
    display(yearly)

    # Store for LLM
    type_specific_summary = {
        'avg_daily_return_pct' : round(avg_return, 4),
        'volatility_pct'       : round(std_return, 4),
        'best_day'             : str(best_day['date'].date()),
        'best_day_return_pct'  : round(best_day['daily_return'], 2),
        'worst_day'            : str(worst_day['date'].date()),
        'worst_day_return_pct' : round(worst_day['daily_return'], 2),
        'positive_days'        : int(positive_days),
        'negative_days'        : int(negative_days),
        'all_time_high'        : round(all_time_high, 2),
        'all_time_high_date'   : str(ath_date.date()),
        'all_time_low'         : round(all_time_low, 2),
        'all_time_low_date'    : str(atl_date.date()),
        'total_return_pct'     : round(total_return, 2),
        'current_price'        : round(latest['close'], 2),
        'trend_signal'         : trend,
        'avg_daily_volume'     : int(avg_vol_shares),
        'peak_volume_date'     : str(max_vol_date2.date()),
        'annual_returns'       : yearly['annual_return_%'].to_dict()
    }


# ════════════════════════════════════════════════════════════
# TRANSACTIONAL EDA
# ════════════════════════════════════════════════════════════
elif dataset_type == 'transactional':
    print('=' * 60)
    print('  TRANSACTIONAL EDA — CREDIT CARD SPEND')
    print('=' * 60)

    amount_col = 'amount'
    date_col   = 'date'

    # ── Overall Spend ────────────────────────────────────────
    total_spend   = df_clean[amount_col].sum()
    avg_txn       = df_clean[amount_col].mean()
    median_txn    = df_clean[amount_col].median()
    max_txn       = df_clean[amount_col].max()
    n_txns        = len(df_clean)

    print(f'\n  Overall Spend:')
    print(f'    Total transactions : {n_txns:,}')
    print(f'    Total spend        : ₹{total_spend:,.0f}')
    print(f'    Avg transaction    : ₹{avg_txn:,.2f}')
    print(f'    Median transaction : ₹{median_txn:,.2f}')
    print(f'    Largest transaction: ₹{max_txn:,.0f}')

    # ── Spend by Category ────────────────────────────────────
    cat_spend = df_clean.groupby('exp_type')[amount_col].agg(['sum','mean','count'])
    cat_spend.columns = ['total_spend', 'avg_spend', 'txn_count']
    cat_spend['spend_pct'] = (cat_spend['total_spend'] / total_spend * 100).round(2)
    cat_spend = cat_spend.sort_values('total_spend', ascending=False)
    print(f'\n  Spend by Category:')
    display(cat_spend)

    # ── Spend by Card Type ───────────────────────────────────
    card_spend = df_clean.groupby('card_type')[amount_col].agg(['sum','count'])
    card_spend.columns = ['total_spend', 'txn_count']
    card_spend['spend_pct'] = (card_spend['total_spend'] / total_spend * 100).round(2)
    card_spend = card_spend.sort_values('total_spend', ascending=False)
    print(f'\n  Spend by Card Type:')
    display(card_spend)

    # ── Spend by Gender ──────────────────────────────────────
    gender_spend = df_clean.groupby('gender')[amount_col].agg(['sum','mean','count'])
    gender_spend.columns = ['total_spend', 'avg_spend', 'txn_count']
    gender_spend['spend_pct'] = (gender_spend['total_spend'] / total_spend * 100).round(2)
    print(f'\n  Spend by Gender:')
    display(gender_spend)

    # ── Monthly Trend ────────────────────────────────────────
    monthly = df_clean.groupby(['year','month'])[amount_col].agg(['sum','count'])
    monthly.columns = ['total_spend', 'txn_count']
    print(f'\n  Monthly Spend Trend (first 12 months):')
    display(monthly.head(12))

    # ── Top Cities ───────────────────────────────────────────
    top_cities = df_clean.groupby('city')[amount_col].sum().sort_values(ascending=False).head(10)
    print(f'\n  Top 10 Cities by Spend:')
    display(top_cities)

    # ── Day of Week Pattern ──────────────────────────────────
    dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    dow_spend = df_clean.groupby('day_of_week')[amount_col].agg(['sum','mean','count'])
    dow_spend.columns = ['total_spend', 'avg_spend', 'txn_count']
    dow_spend = dow_spend.reindex([d for d in dow_order if d in dow_spend.index])
    print(f'\n  Spend by Day of Week:')
    display(dow_spend)

    # Store for LLM
    type_specific_summary = {
        'total_transactions' : n_txns,
        'total_spend'        : round(total_spend, 2),
        'avg_transaction'    : round(avg_txn, 2),
        'median_transaction' : round(median_txn, 2),
        'top_category'       : cat_spend.index[0],
        'top_category_pct'   : round(cat_spend['spend_pct'].iloc[0], 2),
        'top_card_type'      : card_spend.index[0],
        'top_city'           : top_cities.index[0],
        'category_breakdown' : cat_spend['spend_pct'].to_dict(),
        'card_breakdown'     : card_spend['spend_pct'].to_dict(),
        'gender_breakdown'   : gender_spend['spend_pct'].to_dict(),
    }


# ════════════════════════════════════════════════════════════
# FUNDAMENTAL EDA
# ════════════════════════════════════════════════════════════
elif dataset_type == 'fundamental':
    print('=' * 60)
    print('  FUNDAMENTAL EDA — COMPANY FINANCIALS')
    print('=' * 60)

    # ── Universe Overview ────────────────────────────────────
    n_companies = len(df_clean)
    n_industries = df_clean['industry'].nunique() if 'industry' in df_clean.columns else 'N/A'
    print(f'\n  Universe: {n_companies} companies across {n_industries} industries')

    # ── Market Cap Distribution ──────────────────────────────
    if 'market_capitalization' in df_clean.columns:
        mc = df_clean['market_capitalization'].dropna()
        print(f'\n  Market Cap (₹ Cr):')
        print(f'    Total market cap : ₹{mc.sum():,.0f} Cr')
        print(f'    Median           : ₹{mc.median():,.0f} Cr')
        print(f'    Largest company  : {df_clean.loc[mc.idxmax(), "name"]} (₹{mc.max():,.0f} Cr)')
        print(f'    Smallest company : {df_clean.loc[mc.idxmin(), "name"]} (₹{mc.min():,.2f} Cr)')

    # ── Profitability ────────────────────────────────────────
    profit_col = next((c for c in ['net_profit','profit_after_tax'] if c in df_clean.columns), None)
    if profit_col:
        profitable     = (df_clean[profit_col] > 0).sum()
        loss_making    = (df_clean[profit_col] < 0).sum()
        profitable_pct = profitable / n_companies * 100
        print(f'\n  Profitability:')
        print(f'    Profitable companies  : {profitable} ({profitable_pct:.1f}%)')
        print(f'    Loss-making companies : {loss_making}')

    # ── Industry Breakdown ───────────────────────────────────
    if 'industry' in df_clean.columns:
        industry_counts = df_clean['industry'].value_counts().head(15)
        print(f'\n  Top 15 Industries by Company Count:')
        display(industry_counts)

        if 'market_capitalization' in df_clean.columns:
            industry_mc = (df_clean.groupby('industry')['market_capitalization']
                          .sum().sort_values(ascending=False).head(10))
            print(f'\n  Top 10 Industries by Total Market Cap (₹ Cr):')
            display(industry_mc)

    # ── Key Metrics Summary ──────────────────────────────────
    key_metrics = [c for c in ['sales','eps','opm','return_on_capital_employed',
                                'market_capitalization'] if c in df_clean.columns]
    if key_metrics:
        print(f'\n  Key Metrics Summary:')
        display(df_clean[key_metrics].describe().round(2))

    # Store for LLM
    type_specific_summary = {
        'n_companies'      : n_companies,
        'n_industries'     : n_industries,
        'profitable_pct'   : round(profitable_pct, 1) if profit_col else 'N/A',
        'loss_making'      : int(loss_making) if profit_col else 'N/A',
        'top_industries'   : industry_counts.head(5).to_dict() if 'industry' in df_clean.columns else {},
    }

print('\n✅ Type-specific EDA complete.')

  FUNDAMENTAL EDA — COMPANY FINANCIALS

  Universe: 4668 companies across 107 industries

  Market Cap (₹ Cr):
    Total market cap : ₹44,464,893 Cr
    Median           : ₹193 Cr
    Largest company  : Reliance Industr (₹2,119,729 Cr)
    Smallest company : Sagar Soya Prod (₹0.05 Cr)

  Profitability:
    Profitable companies  : 3637 (77.9%)
    Loss-making companies : 1008

  Top 15 Industries by Company Count:


industry
Finance & Investments                             560
Trading                                           532
Miscellaneous                                     410
Construction                                      323
Computers - Software - Medium / Small             241
Chemicals                                         139
Engineering                                       125
Textiles - Products                               118
Auto Ancillaries                                  110
Steel - Medium / Small                            104
Food - Processing - Indian                        100
Entertainment / Electronic Media Software          91
Pharmaceuticals - Indian - Bulk Drugs & Formln     88
Plastics Products                                  87
Electric Equipment                                 84
Name: count, dtype: int64


  Top 10 Industries by Total Market Cap (₹ Cr):


industry
Finance & Investments                             3737484.01
Banks - Private Sector                            3485462.53
Computers - Software - Large                      3226564.17
Refineries                                        2612404.24
Power Generation And Supply                       1967562.00
Miscellaneous                                     1891167.03
Banks - Public Sector                             1698471.48
Construction                                      1432041.47
Pharmaceuticals - Indian - Bulk Drugs & Formln    1401125.39
Trading                                           1220920.62
Name: market_capitalization, dtype: float64


  Key Metrics Summary:


,sales,eps,opm,return_on_capital_employed,market_capitalization
count,4664.00,4640.00,4367.00,4451.00,4668.00
mean,3676.53,20.68,-89.54,15.90,9525.47
std,30030.07,193.53,2264.93,293.50,59237.17
min,-20.20,-486.41,-102800.00,-1535.00,0.05
25%,13.31,0.06,2.78,2.49,36.63
50%,107.82,2.70,9.91,9.60,193.27
75%,731.80,12.05,19.28,18.66,1717.79
max,901064.00,8787.00,3094.12,18600.00,2119728.90



✅ Type-specific EDA complete.


In [29]:
# ── Cell 7: EDA Summary Dictionary ───────────────────────────────────────────
# Packages everything computed above into one clean dictionary.
# This is what gets passed to the LLM in 05_llm_insights.ipynb.
# The LLM never sees raw data — only this structured summary.

eda_summary = {
    'filename'         : DATASET_FILENAME,
    'dataset_type'     : dataset_type,
    'shape'            : df_clean.shape,
    'columns'          : list(df_clean.columns),
    'descriptive_stats': {
        col: {
            'mean'  : round(vals.get('mean', 0), 4),
            'std'   : round(vals.get('std',  0), 4),
            'min'   : round(vals.get('min',  0), 4),
            'max'   : round(vals.get('max',  0), 4),
            'median': round(vals.get('50%',  0), 4),
        }
        for col, vals in desc_stats.items()
    },
    'top_correlations' : top_correlations,
    'domain_analysis'  : type_specific_summary,
}

print('=' * 60)
print('  EDA SUMMARY')
print('=' * 60)
print(f'  Dataset      : {eda_summary["filename"]}')
print(f'  Type         : {eda_summary["dataset_type"].upper()}')
print(f'  Shape        : {eda_summary["shape"]}')
print(f'  Stat columns : {list(eda_summary["descriptive_stats"].keys())}')
print(f'  Top corr     : {eda_summary["top_correlations"][0] if eda_summary["top_correlations"] else "N/A"}')
print(f'\n  Domain analysis keys:')
for k, v in eda_summary['domain_analysis'].items():
    print(f'    {k:<30}: {v}')
print('=' * 60)
print('\n✅ EDA complete. Pass eda_summary into 05_llm_insights.ipynb')

  EDA SUMMARY
  Dataset      : Annual_P_L_1_final.csv
  Type         : FUNDAMENTAL
  Shape        : (4668, 58)
  Stat columns : ['bse_code', 'current_price', 'sales', 'opm', 'profit_after_tax', 'return_on_capital_employed', 'eps', 'change_in_promoter_holding', 'sales_last_year', 'operating_profit_last_year', 'other_income_last_year', 'ebidt_last_year', 'depreciation_last_year', 'ebit_last_year', 'interest_last_year', 'profit_before_tax_last_year', 'tax_last_year', 'profit_after_tax_last_year', 'extraordinary_items_last_year', 'net_profit_last_year', 'dividend_last_year', 'material_cost_last_year', 'employee_cost_last_year', 'opm_last_year', 'npm_last_year', 'operating_profit', 'interest', 'depreciation', 'eps_last_year', 'ebit', 'net_profit', 'current_tax', 'tax', 'other_income', 'last_annual_result_date', 'sales_preceding_year', 'operating_profit_preceding_year', 'other_income_preceding_year', 'ebidt_preceding_year', 'depreciation_preceding_year', 'ebit_preceding_year', 'interest_prec